## Audio Preprocessing and Clipping

In order for the data to be viable for training, it needs to have uniform dimensions. Larger bird classifiers use 5 second clips for training, but its been show that most bird calls can be captured in 2 second clips.

This notebook:
1. Decode -> Mono -> Resample.
2. Caps long recordings by selecting a 30s region.
3. Splits into consecutive 3s windows.
4. Uses a RMS gate to remove silent windows.
5. Runs an existing large bird classifier (BirdNET) as a teacher to label windows as: 
   - target **species** (keep)
   - **non_bird** (keep)
   - **wrong_bird** (drop)
6. Saves a number of species clips and non-bird clips.
7. Writes to manifests.

#### Ensure all dependencies are installed (requirements.txt) 

### Imports and Configs

In [1]:
import sys, platform
import math
import numpy as np
import librosa
import tempfile
import random
import os
import re
import tempfile
import soundfile as sf
import pandas as pd
from birdnetlib.analyzer import Analyzer
from birdnetlib import Recording
from tqdm.auto import tqdm
from pathlib import Path

c:\Users\shado\Year3Projects\FYP\.venv\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
c:\Users\shado\Year3Projects\FYP\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "config.py").exists():
    repo_root = repo_root.parent
assert (repo_root / "src" / "config.py").exists(), f"Couldn't find src/config.py from {Path.cwd()}"

from src.config import CONFIG

cfg = CONFIG.preprocessing

In [3]:
print(CONFIG.preprocessing.clip_len_s)

3.0


### Setup BirdNet teacher
Shoutout to BirdNet for making this really easy

In [10]:
analyzer = Analyzer()

if analyzer is None:
        raise RuntimeError("Teacher analyzer not initialized.")

Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.


c:\Users\shado\Year3Projects\FYP\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


### Utilities

__Root Mean Square (RMS) measures the average signal power over time.__

In [ ]:
def rms_dbfs(x: np.ndarray) -> float:
    """Return RMS loudness in a dBFS-like scale for a mono waveform."""

    rms = float(np.sqrt(np.mean(np.square(x)) + cfg.eps))
    return float(20.0 * math.log10(rms + cfg.eps))


In [ ]:
def build_window_start_times(region_len_s: float) -> list[float]:
    """Return window start times in seconds for a region."""

    start_times = []
    s = cfg.skip_first_s

    while s + cfg.clip_len_s <= region_len_s + 1e-9:
        start_times.append(float(s))
        s += cfg.stride_s

    return start_times


In [ ]:
def load_audio_segment(path: Path, sr: int, offset_s: float, duration_s: float) -> np.ndarray:
    """Load a slice of audio and return it as a mono numpy array."""

    y, _ = librosa.load(str(path), sr=sr, mono=True, offset=float(offset_s), duration=float(duration_s))
    return y


In [ ]:
def save_clip(y_16k: np.ndarray, out_dir: Path, xc_id: str, start_s: float, end_s: float) -> str:
    """Save a 16-bit PCM WAV clip and return its path."""

    start_ms = int(round(start_s * 1000))
    end_ms = int(round(end_s * 1000))
    fname = f"XC{xc_id}__s{start_ms}__e{end_ms}.wav"

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / fname
    sf.write(str(out_path), y_16k, cfg.sample_rate_model, subtype="PCM_16")
    return str(out_path)


In [ ]:
def choose_best_region(path: Path, total_len_s: float) -> tuple[float, np.ndarray]:
    """Pick the most active region for long recordings.

    If the recording is shorter than the cap, return the full audio. Otherwise,
    slide a fixed window of length cfg.recording_cap_s every cfg.step_cap_s and
    choose the region with the highest RMS.
    """

    if total_len_s <= cfg.recording_cap_s + 1e-9:
        y = load_audio_segment(path, cfg.sample_rate_model, 0.0, total_len_s)
        return 0.0, y

    best_start = 0.0
    best_score = -float("inf")
    best_y = None
    max_start = max(0.0, total_len_s - cfg.recording_cap_s)

    for s in np.arange(0.0, max_start + 1e-9, cfg.step_cap_s):
        y = load_audio_segment(path, cfg.sample_rate_model, float(s), cfg.recording_cap_s)
        score = rms_dbfs(y)
        if score > best_score:
            best_start = float(s)
            best_score = score
            best_y = y

    if best_y is None:
        best_y = load_audio_segment(path, cfg.sample_rate_model, 0.0, min(total_len_s, cfg.recording_cap_s))
        best_start = 0.0

    return best_start, best_y


### Using the BirdNet Teacher
BirdNET is used as an offline teacher model to automatically validate and label candidate 3-second audio clips. After RMS-based energy gating removes silent windows, each remaining clip is resampled to 48 kHz and analyzed with BirdNET. Based on the top detection and its confidence, each clip is classified as species (target species detected with high confidence), non_bird (no bird detected), or drop (a bird detected but not the target species). Only a small, randomly selected subset of species and non_bird clips per recording is retained.

In [12]:
def teacher_analyze_window(y_16k: np.ndarray, target_sci_name: str) -> dict:
    """Run BirdNET on a 3s window and return a decision and detection metadata."""

    # BirdNET expects 48 kHz input audio.
    y_48k = librosa.resample(y_16k, orig_sr=cfg.sample_rate_model, target_sr=cfg.sample_rate_teacher)

    # Creating a temp file as input
    fd, tmp_path = tempfile.mkstemp(suffix=".wav")
    os.close(fd)

    try:
        sf.write(tmp_path, y_48k, cfg.sample_rate_teacher, subtype="PCM_16")

        rec = Recording(analyzer, tmp_path, min_conf=cfg.bird_conf_thr)
        rec.analyze()
        detections = rec.detections or []
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass

    # Pick the top detection by confidence for the decision.
    top = None
    max_conf = 0.0
    for d in detections:
        c = float(d.get("confidence", 0.0))
        if c > max_conf:
            max_conf, top = c, d

    top_sci = (top.get("scientific_name") if top else "")
    top_common = (top.get("common_name") if top else "")
    top_conf = float(top.get("confidence", 0.0)) if top else 0.0

    target_norm = target_sci_name.strip().lower()
    is_target = bool(top_sci) and (top_sci.strip().lower() == target_norm) and (top_conf >= cfg.species_conf_thr)

    if len(detections) == 0:
        decision = "non_bird"
    elif is_target:
        decision = "species"
    else:
        decision = "drop"

    return {
        "detections": detections,
        "top_sci": top_sci,
        "top_common": top_common,
        "top_conf": top_conf,
        "max_conf": float(max_conf),
        "decision": decision,
    }


### Running the Pipeline